In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!git clone --quiet --recursive https://github.com/cvg/Hierarchical-Localization/
%cd Hierarchical-Localization
!pip install --progress-bar off --quiet -e .
!pip install --progress-bar off --quiet --upgrade plotly

from tqdm.notebook import tqdm
from pathlib import Path

from hloc import extract_features, match_features, reconstruction, visualization, pairs_from_exhaustive
from hloc.visualization import plot_images, read_image
from hloc.utils import viz_3d

fatal: destination path 'Hierarchical-Localization' already exists and is not an empty directory.


/content/Hierarchical-Localization
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:
DATA_PATH = Path('/content/drive/MyDrive/buggy_hloc')

In [7]:
videos = list(DATA_PATH.glob('*.mp4'))
videos

[PosixPath('/content/drive/MyDrive/buggy_hloc/37.mp4'),
 PosixPath('/content/drive/MyDrive/buggy_hloc/38.mp4'),
 PosixPath('/content/drive/MyDrive/buggy_hloc/45.mp4'),
 PosixPath('/content/drive/MyDrive/buggy_hloc/1388.mp4'),
 PosixPath('/content/drive/MyDrive/buggy_hloc/1401.mp4')]

In [9]:
import cv2

STRIDE = 1  # keep every Nth frame
IMAGES_PATH = DATA_PATH / 'images'

for video in tqdm(videos, desc='videos'):
    out_dir = IMAGES_PATH / video.stem
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    saved = 0
    i = 0
    try:
        with tqdm(total=frame_count, desc=video.stem, leave=False) as progress:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                i += 1
                progress.update(1)
                if i % STRIDE != 0:
                    continue
                cv2.imwrite(str(out_dir / f"{i:06d}.jpg"), frame)
                saved += 1
    finally:
        cap.release()
    print(f"{video.stem}: saved {saved} frames to {out_dir}")

videos:   0%|          | 0/5 [00:00<?, ?it/s]

37:   0%|          | 0/10401 [00:00<?, ?it/s]

37: saved 10401 frames to /content/drive/MyDrive/buggy_hloc/images/37


38:   0%|          | 0/6249 [00:00<?, ?it/s]

38: saved 6249 frames to /content/drive/MyDrive/buggy_hloc/images/38


...

45: saved 8259 frames to /content/drive/MyDrive/buggy_hloc/images/45


1388:   0%|          | 0/7714 [00:00<?, ?it/s]

1388: saved 7714 frames to /content/drive/MyDrive/buggy_hloc/images/1388


1401:   0%|          | 0/12090 [00:00<?, ?it/s]

1401: saved 12090 frames to /content/drive/MyDrive/buggy_hloc/images/1401
